# Module 03 — Tokenization (notebook)

Walkthrough of [`bpe_from_scratch.py`](bpe_from_scratch.py), [`train_bpe.py`](train_bpe.py), and [`tokenizer.py`](tokenizer.py). We'll:

1. Watch BPE learn merges, one step at a time, on a paragraph.
2. Train a real (small) BPE on FineWeb-Edu using the production code path.
3. Roundtrip-test encode/decode.
4. Compare compression rates against GPT-2 and Qwen 3 on English, code, and Chinese.
5. See the digit-tokenization and whitespace failure modes for ourselves.

**Compute:** CPU-only. ~3–5 minutes total. The slow step is streaming a few thousand FineWeb-Edu docs.

**Prereqs:** Module 02 set up (we import from `02-the-corpus/corpus.py`).

In [ ]:
import sys
from itertools import islice
from pathlib import Path

# Let us import corpus.py from the sibling Module 02 folder.
sys.path.insert(0, str(Path.cwd().parent / "02-the-corpus"))

import bpe_from_scratch
from corpus import stream_fineweb_edu
from train_bpe import train_bpe
from tokenizer import compression_rate

## 1. BPE, one merge at a time

Run the from-scratch trainer on a small paragraph and print every merge as it happens. You'll see BPE first grab the highest-frequency byte pairs (often `e ` or `he `), then progressively build up common words.

In [ ]:
sample = (
    "Tokenization turns text into integers. "
    "Integers are what neural networks know how to multiply. "
    "Tokenization is therefore where every language model begins. "
    "Tokenization choices constrain everything downstream."
)
vocab, merges = bpe_from_scratch.train(sample, vocab_size=276, verbose=True)
print(f"\nfinal vocab size: {len(vocab)}, merges learned: {len(merges)}")

Read the merges. Early ones pick up the most-frequent byte bigrams (often space-followed-by-letter or letter-followed-by-space). Later merges build up common words like `Tokenization`. This is exactly the algorithm Llama, Qwen, and DeepSeek use — just run on trillions of characters with a Rust-based implementation.

## 2. Train a real BPE on FineWeb-Edu

Same algorithm, real corpus, the HuggingFace `tokenizers` library. We use a small slice (2000 docs, 8k vocab) so this finishes in ~1–2 minutes; the canonical training run (32k vocab, 100k docs) is what `train_bpe.py`'s CLI does in `~10` minutes.

In [ ]:
NB_TOKENIZER_PATH = Path.cwd() / "results" / "tokenizer_notebook.json"

stream = stream_fineweb_edu(min_score=3.0, shuffle_buffer=500, seed=42)
text_iter = (doc["text"] for doc in islice(stream, 2000))

tok = train_bpe(
    text_iter,
    vocab_size=8_000,
    output_path=NB_TOKENIZER_PATH,
    show_progress=False,
)
print(f"trained tokenizer with vocab_size = {tok.get_vocab_size()}")
print(f"saved to {NB_TOKENIZER_PATH}")

## 3. Encode, decode, roundtrip

Verify the basic contract: encoding then decoding gives you the original string back.

In [ ]:
test = "The quick brown fox jumps over the lazy dog. Tokenization is the bridge."
encoded = tok.encode(test)
print(f"text  ({len(test)} chars): {test!r}")
print(f"ids   ({len(encoded.ids)} tokens): {encoded.ids[:20]}{'...' if len(encoded.ids) > 20 else ''}")
print(f"tokens:               {encoded.tokens[:20]}{'...' if len(encoded.tokens) > 20 else ''}")
decoded = tok.decode(encoded.ids)
print(f"roundtrip ok:         {decoded == test}")
print(f"compression:          {compression_rate(tok, test):.2f} chars/token")

Note the `tokens` field — these are the human-readable surface forms. Spaces show as `Ġ` (the byte-level encoding convention for the space character). That's a feature, not a bug: it lets BPE distinguish `hello` (word-internal) from `Ġhello` (word-initial).

## 4. How does our tokenizer compare to the big ones?

Load GPT-2 (50k vocab, English-centric) and Qwen 3 (152k vocab, multilingual) for comparison. Tokenizers are small files (~5–20 MB each); they download on first use.

In [ ]:
from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained("gpt2")
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

print(f"ours (FineWeb-Edu 8k): vocab = {tok.get_vocab_size():>7,}")
print(f"GPT-2:                 vocab = {gpt2.vocab_size:>7,}")
print(f"Qwen 3:                vocab = {qwen.vocab_size:>7,}")

In [ ]:
def chars_per_token(name, encode_fn, text):
    n_tok = len(encode_fn(text))
    return f"{name:12s} {len(text):6d} chars -> {n_tok:5d} tokens  ({len(text)/n_tok:.2f} chars/tok)"

def measure(text, label):
    print(f"\n--- {label} ({len(text)} chars) ---")
    print(chars_per_token("ours (8k)", lambda t: tok.encode(t).ids, text))
    print(chars_per_token("GPT-2", lambda t: gpt2.encode(t), text))
    print(chars_per_token("Qwen 3", lambda t: qwen.encode(t), text))

In [ ]:
english = (
    "The transformer architecture has dominated language modeling since 2017. "
    "Most innovations since then have been refinements: better attention variants, "
    "smarter normalization, mixture of experts, learned position encodings. "
    "The basic recipe — multi-head self-attention plus feedforward, stacked — has held."
)
measure(english, "English prose")

Our 8k tokenizer is competitive on English — it was trained on FineWeb-Edu, which is exactly this kind of text. GPT-2 (50k) edges it out slightly because it has more room. Qwen 3 (152k) is similar, since its English compression doesn't really benefit from the extra vocab budget that's spent on Chinese.

Now Chinese:

In [ ]:
chinese = (
    "语言模型是人工智能的核心技术之一。"
    "通过在大量文本上训练神经网络，模型可以学习生成连贯的语言。"
    "分词器决定了模型如何看待文本。"
    "对于中文，分词器需要为常见的汉字和词组分配独立的标记。"
)
measure(chinese, "Chinese prose")

**This is the multilingual gap.** Our tokenizer and GPT-2 both score under 1 char/token on Chinese (each character costs 2–3 byte tokens), while Qwen 3 hits 1.5+ chars/token because it has explicit Chinese-character tokens. For a Chinese user, training with our tokenizer would mean 3–4× longer sequences than necessary — a permanent training and inference tax.

Now code:

In [ ]:
code = '''def attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = scores.softmax(dim=-1)
    return weights @ v
'''
measure(code, "Python code")

GPT-2 and Qwen both do well on code (their training corpora included GitHub); our FineWeb-Edu tokenizer is worse because the educational web corpus has comparatively little code. **Domain-mismatched tokenizers cost you forever**, in exactly this way.

## 5. Digit tokenization

We trained with `Digits(individual_digits=True)` so every digit is its own token. GPT-2 didn't — it freely merged multi-digit numbers into single tokens. Watch:

In [ ]:
for number in ["1234", "2026", "31415926", "100000"]:
    our_tokens = tok.encode(number).tokens
    gpt2_tokens = gpt2.tokenize(number)
    print(f"{number:12s}  ours: {our_tokens}")
    print(f"{'':12s}  gpt2: {gpt2_tokens}")

GPT-2 happily turns `100000` into one or two tokens. The model then has to memorize that this single token *means* the same thing as a particular position in a number sequence — which is why GPT-3 and GPT-3.5 were famously bad at arithmetic. Modern recipes (Llama 3, DeepSeek, our pretokenizer) force one token per digit so the model sees digit positions consistently. Arithmetic accuracy goes up substantially with no other changes.

## 6. The whitespace surprise

`"hello"` and `" hello"` are not the same tokens. Many prompt-engineering bugs live here.

In [ ]:
for s in ["hello", " hello", "hello world", " hello world"]:
    enc = tok.encode(s)
    print(f"{s!r:20s} ids={enc.ids}  tokens={enc.tokens}")

The leading-space variant tokenizes the space *as part of* the first token (e.g. `Ġhello` instead of `hello`). They are different token IDs, and the model assigns them different probabilities. If your prompt accidentally ends with a space, you'll get unexpected continuations.

## Recap

You can now:

- Implement BPE from scratch and explain what each step does.
- Train a real byte-level BPE on a real corpus using `train_bpe.py`.
- Measure compression rate and reason about why one tokenizer outperforms another on a given language or domain.
- Recognize the digit, whitespace, and multilingual failure modes.
- Defend a vocab-size choice for a given model and corpus.

**Next:** [Part 2 — Architecture](../../part-2-architecture/). The tokens are integers now. The next twenty thousand decisions are about what to compute with them — starting with attention.